# Hotel Booking Cancellation Prediction

**Tabular Regression Project** — Predict hotel booking metrics.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Hotel Booking Demand (Kaggle)
Target: `adr` (Average Daily Rate)

## 2 · Project Overview

We predict the **Average Daily Rate (ADR)** of hotel bookings using reservation features. ADR is a key hotel revenue metric — predicting it helps with revenue management, pricing strategy, and demand forecasting.

The dataset contains ~120K bookings with rich features about guest demographics, booking details, and hotel operations.

## 3 · Learning Objectives

1. Work with a large, feature-rich hotel dataset.
2. Handle date-based and categorical features.
3. Apply regression models to revenue management.
4. Use LazyPredict and FLAML for model selection.
5. Understand hotel industry KPIs (ADR, RevPAR).

## 4 · Problem Statement

Predict the **Average Daily Rate (ADR)** for hotel bookings based on reservation details, guest information, and hotel characteristics.

## 5 · Why This Project Matters

- **Revenue management** is the most profitable application of ML in hospitality.
- ADR prediction directly impacts room pricing, forecasting, and profitability.
- Hotels use these models for dynamic pricing across seasons and customer segments.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Kaggle: jessemostipak/hotel-booking-demand |
| **Rows** | ~119K |
| **Target** | adr (Average Daily Rate, EUR) |
| **Features** | hotel, lead_time, arrival_date, stays, guests, meal, country, market_segment, etc. |

## 7 · Dataset Source and License Notes

- **Source**: Hotel booking demand dataset from ScienceDirect article.
- **License**: Open for research/educational use.
- **Note**: Real hotel booking data from Portugal (City Hotel + Resort Hotel).

## 8 · Environment Setup

In [ ]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
_install_if_missing('kagglehub')
print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'adr'
MAX_ROWS = 50000
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
import kagglehub, glob

try:
    path = kagglehub.dataset_download('jessemostipak/hotel-booking-demand')
    print(f'Downloaded to: {path}')
except Exception as e:
    print(f'Download error: {e}')
    path = SAVE_DIR

csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV found in {path}'
df = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
print(f'Loaded: {df.shape}')
if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f'Sampled to: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values (top 10):')
print(df.isnull().sum().sort_values(ascending=False).head(10))
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'\nNegative ADR: {(df[TARGET] < 0).sum()}')

# Remove negative/zero ADR rows
df = df[df[TARGET] > 0].reset_index(drop=True)
print(f'After cleanup: {df.shape}')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df[TARGET].hist(bins=50, ax=axes[0,0], edgecolor='black')
axes[0,0].set_title('ADR Distribution')

if 'hotel' in df.columns:
    df.groupby('hotel')[TARGET].mean().plot.bar(ax=axes[0,1])
    axes[0,1].set_title('Avg ADR by Hotel Type')

if 'arrival_date_month' in df.columns:
    month_order = ['January','February','March','April','May','June','July','August','September','October','November','December']
    monthly = df.groupby('arrival_date_month')[TARGET].mean()
    monthly = monthly.reindex([m for m in month_order if m in monthly.index])
    monthly.plot.bar(ax=axes[1,0])
    axes[1,0].set_title('Avg ADR by Month')
    plt.setp(axes[1,0].get_xticklabels(), rotation=45)

if 'lead_time' in df.columns:
    axes[1,1].scatter(df['lead_time'], df[TARGET], alpha=0.1, s=1)
    axes[1,1].set_xlabel('Lead Time'); axes[1,1].set_ylabel(TARGET)
    axes[1,1].set_title('Lead Time vs ADR')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Drop is_canceled (classification target) if present
if 'is_canceled' in df.columns:
    df = df.drop(columns=['is_canceled'])

# Drop reservation_status (leakage)
if 'reservation_status' in df.columns:
    df = df.drop(columns=['reservation_status'])
if 'reservation_status_date' in df.columns:
    df = df.drop(columns=['reservation_status_date'])

# Encode categoricals
for col in df.select_dtypes(include='object').columns:
    if df[col].nunique() > 100:
        df = df.drop(columns=[col])
    else:
        df[col] = df[col].fillna('Unknown')
        df[col] = LabelEncoder().fit_transform(df[col])

for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

print(f'Preprocessed: {df.shape}')

## 17 · Feature Engineering

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R²: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Boosting Models

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Eval of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation and Business Insight

- **Hotel type** (City vs Resort) creates distinct pricing tiers.
- **Arrival month** captures seasonal demand patterns — summer/holidays drive higher ADR.
- **Lead time** and **market segment** influence willingness to pay.
- Revenue managers can use these models for dynamic pricing strategies.

## 26 · Limitations

- Data from two Portuguese hotels — limited geographic generalization.
- No competitor pricing or external event data.
- ADR is per-booking, not per-room-night revenue.
- Market conditions change rapidly in hospitality.

## 27 · How to Improve

1. Add external data: local events, holidays, weather.
2. Include competitor pricing.
3. Time-series features for demand trends.
4. Segment-specific models (business vs leisure).
5. Add day-of-week features.

## 28 · Production Considerations

- Real-time integration with PMS (Property Management System).
- Dynamic pricing updates multiple times per day.
- A/B testing of pricing strategies.
- Revenue management requires human oversight.

## 29 · Common Mistakes

1. Including reservation_status (leakage — known after outcome).
2. Including is_canceled as a feature (classification label).
3. Not removing negative ADR rows.
4. Ignoring the seasonal patterns in hotel pricing.

## 30 · Mini Challenge

1. Build separate models for City Hotel and Resort Hotel.
2. Add total_stay = stays_weekend + stays_weekday.
3. Try log-transforming ADR.
4. Create a 'high_season' binary feature.

## 31 · Final Summary

- Hotel ADR prediction is a high-value revenue management application.
- Seasonal patterns and hotel type are dominant pricing drivers.
- Data cleaning (removing leakage, handling negative ADR) is crucial.
- Gradient-boosting models handle the mixed feature types effectively.

In [ ]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')